In [1]:
!pip uninstall -y torch torchvision torchaudio
!pip install -q torch torchvision torchaudio
!pip install -q transformers sentencepiece accelerate

Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90

In [2]:
import torch
print(torch.__version__)

2.14.1+cu130


In [3]:
from transformers import BartTokenizer,BartForConditionalGeneration
import re

model_name="facebook/bart-large-cnn"

tokenizer=BartTokenizer.from_pretrained(model_name)
model=BartForConditionalGeneration.from_pretrained(model_name)

print("Model loaded successfully!")

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded successfully!


text preprocessing

In [4]:
def preprocess_text(text):

  text=re.sub(r'\s+',' ',text)

  text=text.strip()
  return text

In [5]:
document="""Artificial Intelligence (AI) is the branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include learning from data, recognizing speech and images, understanding language, solving problems, and making decisions. AI is widely used in everyday applications such as virtual assistants, recommendation systems, self-driving technology, healthcare, education, and finance, helping improve efficiency and accuracy.

AI offers many benefits, including faster decision-making, automation of repetitive tasks, and improved productivity across industries. However, it also presents challenges such as concerns about privacy, data security, job displacement, and ethical use. As AI continues to advance, it is important to develop and use it responsibly to ensure that it benefits society while minimizing potential risks."""

document=document * 20


In [6]:
document=preprocess_text(document)

print("Preprocessing Completed")

Preprocessing Completed


In [9]:

# Step 1: Tokenize the document
inputs = tokenizer(
    document,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
)

print("Document Tokenized Successfully!")

# Step 2: Generate summary
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=100,
    min_length=30,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Generated Summary:")
print(summary)


# Step 3: Split large text into chunks
def split_text(text, chunk_size=500):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


# Step 4: Summarize a large document
def summarize_large_document(text):
    chunks = split_text(text)
    summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            max_length=1024,
            truncation=True
        )

        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=80,
            min_length=20,
            num_beams=4
        )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        summaries.append(summary)

    final_summary = " ".join(summaries)
    return final_summary


# Step 5: Display the final summary
final_summary = summarize_large_document(document)

print("Final Summary:\n")
print(final_summary)


Document Tokenized Successfully!
Generated Summary:
Artificial Intelligence (AI) is the branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include learning from data, recognizing speech and images, understanding language, solving problems, and making decisions. AI is widely used in everyday applications such as virtual assistants, recommendation systems, self-driving technology, healthcare, education, and finance. AI offers many benefits, including faster decision-making, automation of repetitive tasks, and improved productivity
Final Summary:

Artificial Intelligence (AI) is the branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include learning from data, recognizing speech and images, understanding language, solving problems, and making decisions. AI is widely used in everyday